# Laboratorio 4 — Inciso 7: Análisis de los lagos y comparación entre ellos

Se compara la proliferación de cianobacteria entre el Lago de Atitlán y el Lago de Amatitlán a lo largo
del período estudiado, usando la serie temporal calculada en el inciso 4 (`serie_temporal_cianobacteria.csv`)
y las correlaciones del inciso 6 (`correlaciones_indices.csv`). No se vuelve a abrir ningún raster: todo
sale de los CSV ya procesados.


In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.config import FECHAS_OFICIALES, LAGOS, NUBOSIDAD_OFICIAL, RUTA_DATA_PROCESSED, RUTA_FIGURAS

ruta_serie = RUTA_DATA_PROCESSED / "serie_temporal_cianobacteria.csv"
ruta_corr = RUTA_DATA_PROCESSED / "correlaciones_indices.csv"
assert ruta_serie.exists(), f"Falta {ruta_serie}: correr primero 04_analisis_temporal.ipynb"
assert ruta_corr.exists(), f"Falta {ruta_corr}: correr primero 06_correlacion.ipynb"

serie_temporal = pd.read_csv(ruta_serie, parse_dates=["fecha"])
correlaciones = pd.read_csv(ruta_corr)
serie_temporal.head()


## 7.1 Proliferación de cianobacteria por lago en el período estudiado

Estadísticos resumen del índice de cianobacteria (`clorofila_promedio`) por lago sobre las 11 fechas
oficiales de cada uno: nivel típico, dispersión y fecha del valor máximo observado.


In [ ]:
def resumen_lago(lago):
    sub = serie_temporal[serie_temporal["lago"] == lago].sort_values("fecha")
    fila_pico = sub.loc[sub["clorofila_promedio"].idxmax()]
    primera, ultima = sub.iloc[0]["clorofila_promedio"], sub.iloc[-1]["clorofila_promedio"]
    return {
        "lago": lago,
        "nombre": LAGOS[lago]["nombre"],
        "n_fechas": len(sub),
        "clorofila_media": sub["clorofila_promedio"].mean(),
        "clorofila_mediana": sub["clorofila_promedio"].median(),
        "clorofila_std": sub["clorofila_promedio"].std(),
        "clorofila_max": fila_pico["clorofila_promedio"],
        "fecha_pico": fila_pico["fecha"].strftime("%Y-%m-%d"),
        "cambio_primera_ultima": ultima - primera,
    }

resumen_por_lago = pd.DataFrame([resumen_lago(lago) for lago in LAGOS]).set_index("lago")
resumen_por_lago


La columna `cambio_primera_ultima` indica si el nivel de cianobacteria subió o bajó entre la primera y la
última fecha oficial del lago; junto con `clorofila_std` describe si la proliferación fue estable, creciente
o errática en el período.


## 7.2 Comparación de intensidad y frecuencia de floraciones entre ambos lagos

**Intensidad**: distribución del índice de cianobacteria en las 11 fechas de cada lago (boxplot).

**Frecuencia**: se define una floración como una fecha cuyo `clorofila_promedio` supera un umbral absoluto
común a ambos lagos (`UMBRAL_FLORACION`), para comparar de forma justa cuántas veces floreció cada uno con
la misma vara. El umbral es ajustable: se fija aquí en el percentil 75 del conjunto combinado de ambos
lagos como punto de partida razonable.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
datos_box = [serie_temporal[serie_temporal["lago"] == lago]["clorofila_promedio"] for lago in LAGOS]
etiquetas = [LAGOS[lago]["nombre"] for lago in LAGOS]
ax.boxplot(datos_box, tick_labels=etiquetas, patch_artist=True,
           boxprops=dict(facecolor="lightseagreen", alpha=0.6))
ax.set_ylabel("Índice de cianobacteria (clorofila_promedio)")
ax.set_title("Intensidad de la cianobacteria por lago (11 fechas oficiales)")
ax.grid(alpha=0.3, axis="y")
fig.tight_layout()
fig.savefig(RUTA_FIGURAS / "distribucion_intensidad_lagos.png", dpi=150)
plt.show()


In [ ]:
# ponytail: umbral absoluto común y ajustable, no es un valor "mágico" fijo en el código;
# cambiar aquí si se prefiere otro criterio de floración.
UMBRAL_FLORACION = serie_temporal["clorofila_promedio"].quantile(0.75)
print(f"UMBRAL_FLORACION = {UMBRAL_FLORACION:.3f}")

frecuencia_filas = []
for lago in LAGOS:
    sub = serie_temporal[serie_temporal["lago"] == lago]
    n_floraciones = int((sub["clorofila_promedio"] > UMBRAL_FLORACION).sum())
    frecuencia_filas.append({
        "lago": lago,
        "nombre": LAGOS[lago]["nombre"],
        "n_fechas": len(sub),
        "n_floraciones": n_floraciones,
        "frecuencia_pct": 100 * n_floraciones / len(sub),
    })
frecuencia_por_lago = pd.DataFrame(frecuencia_filas).set_index("lago")
frecuencia_por_lago


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.bar(frecuencia_por_lago["nombre"], frecuencia_por_lago["frecuencia_pct"], color="darkorange", alpha=0.75)
for i, (_, fila) in enumerate(frecuencia_por_lago.iterrows()):
    ax.text(i, fila["frecuencia_pct"] + 1, f"{fila['n_floraciones']}/{fila['n_fechas']} fechas", ha="center")
ax.set_ylabel("% de fechas con floración (umbral común)")
ax.set_title(f"Frecuencia de floraciones por lago (umbral = {UMBRAL_FLORACION:.2f})")
ax.set_ylim(0, 100)
ax.grid(alpha=0.3, axis="y")
fig.tight_layout()
fig.savefig(RUTA_FIGURAS / "frecuencia_floraciones_lagos.png", dpi=150)
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
colores = {"atitlan": "teal", "amatitlan": "darkorange"}
for lago in LAGOS:
    sub = serie_temporal[serie_temporal["lago"] == lago].sort_values("fecha")
    ax.plot(sub["fecha"], sub["clorofila_promedio"], marker="o", label=LAGOS[lago]["nombre"], color=colores[lago])
ax.axhline(UMBRAL_FLORACION, color="red", linestyle="--", alpha=0.6, label=f"Umbral de floración ({UMBRAL_FLORACION:.2f})")
ax.set_ylabel("Índice de cianobacteria (clorofila_promedio)")
ax.set_xlabel("Fecha")
ax.set_title("Evolución de la cianobacteria en ambos lagos")
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(RUTA_FIGURAS / "comparacion_temporal_ambos_lagos.png", dpi=150)
plt.show()


In [ ]:
comparacion_lagos = resumen_por_lago.join(frecuencia_por_lago[["n_floraciones", "frecuencia_pct"]])
comparacion_lagos.to_csv(RUTA_DATA_PROCESSED / "comparacion_lagos.csv")
comparacion_lagos


### Self-check

Verificación mínima de que la tabla de comparación es internamente consistente antes de usarla en el
informe: las columnas esperadas existen, el número de fechas coincide con `FECHAS_OFICIALES` y el conteo
de floraciones nunca excede el número de fechas del lago.


In [ ]:
columnas_esperadas = {"n_fechas", "clorofila_media", "clorofila_max", "fecha_pico", "n_floraciones", "frecuencia_pct"}
assert columnas_esperadas.issubset(comparacion_lagos.columns), "faltan columnas en comparacion_lagos"

for lago in LAGOS:
    fila = comparacion_lagos.loc[lago]
    assert fila["n_fechas"] == len(FECHAS_OFICIALES[lago]), f"{lago}: n_fechas no coincide con FECHAS_OFICIALES"
    assert 0 <= fila["n_floraciones"] <= fila["n_fechas"], f"{lago}: n_floraciones fuera de rango"

print("OK: comparacion_lagos.csv es consistente.")
